### Start Spark

In [1]:
# Import the tool that manages our connection to the Spark engine
from pyspark.sql import SparkSession

# Import specific calculation/helper tools we'll need throughout:
# col = refer to a column, lit = a fixed/literal value, avg = average,
# count = count rows, round = round a decimal number
from pyspark.sql.functions import col, lit, avg, count, round

# Start (or reuse) a Spark session, and give it a name for identification
spark = SparkSession.builder.appName("SnackAnalysis").getOrCreate()

# Hide Spark's noisy informational log messages, keep output clean
spark.sparkContext.setLogLevel("WARN")

### Load the 4 Excel files safely

In [2]:
import pandas as pd

def load_excel_safe(filename, sheet_name=0):
    # Read the Excel file into a pandas table first,
    # since PySpark cannot open .xlsx files directly
    pdf = pd.read_excel(filename, sheet_name=sheet_name)

    # Convert every single value in the table to plain text.
    # This avoids PySpark's mixed-type errors (e.g. some rows being
    # whole numbers, others decimals, or a stray date hiding in a
    # numeric column) - we convert the needed columns back to
    # proper numbers explicitly, later, in a controlled way.
    pdf = pdf.map(lambda x: str(x) if pd.notna(x) else None)

    # Convert the now-all-text pandas table into a PySpark table
    return spark.createDataFrame(pdf)

# Load all 4 source files
mc_raw = load_excel_safe("menu.xlsx")                               # McDonald's full menu
sb_drinks1_raw = load_excel_safe("starbucks_drinkMenu_expanded.xlsx") # Starbucks drinks (detailed)
sb_drinks2_raw = load_excel_safe("starbucks-menu-nutrition-drinks.xlsx") # Starbucks drinks (2nd source)
sb_food_raw = load_excel_safe("starbucks-menu-nutrition-food.xlsx")  # Starbucks food

print("Loaded all 4 files")

Loaded all 4 files


### Clean, rename, and type-convert each file's columns

In [6]:
from pyspark.sql.functions import expr

def try_double(colname):
    # Same as col(colname).cast("double"), but tolerates malformed
    # values (like a stray date hiding in a numeric column) by
    # converting them to NULL instead of crashing the whole operation
    return expr(f"try_cast(`{colname}` as double)")

# --- McDonald's ---
mcd_clean = mc_raw.select(
    col("Item").alias("Item_Name"),
    try_double("Calories").alias("Calories"),
    try_double("Total Fat").alias("Fat_g"),
    try_double("Carbohydrates").alias("Carbs_g"),
    try_double("Dietary Fiber").alias("Fiber_g"),
    try_double("Protein").alias("Protein_g"),
    try_double("Sodium").alias("Sodium_mg"),
    try_double("Sugars").alias("Sugar_g"),
).withColumn("Brand", lit("McDonald's")).withColumn("item_type", lit("Food"))


# --- Starbucks drinks file 1 (the detailed one) ---
sb_drinks1_clean = sb_drinks1_raw.select(
    col("Beverage").alias("Item_Name"),
    try_double("Calories").alias("Calories"),
    try_double("Total Fat (g)").alias("Fat_g"),
    try_double("Total Carbohydrates (g)").alias("Carbs_g"),
    try_double("Dietary Fibre (g)").alias("Fiber_g"),
    try_double("Protein (g)").alias("Protein_g"),
    try_double("Sodium (mg)").alias("Sodium_mg"),
    try_double("Sugars (g)").alias("Sugar_g"),
).withColumn("Brand", lit("Starbucks")).withColumn("item_type", lit("Drink"))


# --- Starbucks drinks file 2 (secondary source, fewer columns) ---
sb_drinks2_clean = sb_drinks2_raw.select(
    col("`Unnamed: 0`").alias("Item_Name"),
    try_double("Calories").alias("Calories"),
    try_double("Fat (g)").alias("Fat_g"),
    try_double("Carb. (g)").alias("Carbs_g"),
    try_double("Fiber (g)").alias("Fiber_g"),
    try_double("Protein").alias("Protein_g"),
    try_double("Sodium").alias("Sodium_mg"),
    lit(None).cast("double").alias("Sugar_g"),
).withColumn("Brand", lit("Starbucks")).withColumn("item_type", lit("Drink"))


# --- Starbucks food file ---
sb_food_clean = sb_food_raw.select(
    col("`Unnamed: 0`").alias("Item_Name"),
    try_double("Calories").alias("Calories"),
    try_double("Fat (g)").alias("Fat_g"),
    try_double("Carb. (g)").alias("Carbs_g"),
    try_double("Fiber (g)").alias("Fiber_g"),
    try_double("Protein (g)").alias("Protein_g"),
    lit(None).cast("double").alias("Sodium_mg"),
    lit(None).cast("double").alias("Sugar_g"),
).withColumn("Brand", lit("Starbucks")).withColumn("item_type", lit("Food"))

print("Cleaned and standardized all 4 datasets")

Cleaned and standardized all 4 datasets


### Combine all 4 into one table

In [7]:
# Stack all 4 cleaned tables on top of each other. unionByName matches
# columns by their name, which works here since every table above now
# has the exact same 8 column names (Item_Name, Calories, Fat_g, etc.)
combined = mcd_clean.unionByName(sb_drinks1_clean).unionByName(sb_drinks2_clean).unionByName(sb_food_clean)

print("Combined total rows:", combined.count())

# Sanity check: confirm the row counts per brand match what we expect
combined.groupBy("Brand").count().orderBy("Brand").show()

Combined total rows: 792
+----------+-----+
|     Brand|count|
+----------+-----+
|McDonald's|  260|
| Starbucks|  532|
+----------+-----+



### Check for missing data before trusting any averages

In [8]:
from pyspark.sql.functions import sum as spark_sum, when

nutrient_cols = ["Calories", "Fat_g", "Carbs_g", "Fiber_g", "Protein_g", "Sodium_mg", "Sugar_g"]

print("--- Missing value counts per column, per brand ---")
combined.groupBy("Brand").agg(
    *[spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in nutrient_cols]
).show()

--- Missing value counts per column, per brand ---
+----------+--------+-----+-------+-------+---------+---------+-------+
|     Brand|Calories|Fat_g|Carbs_g|Fiber_g|Protein_g|Sodium_mg|Sugar_g|
+----------+--------+-----+-------+-------+---------+---------+-------+
|McDonald's|       0|    0|      0|      0|        0|        0|      0|
| Starbucks|      85|   86|     85|     85|       85|      198|    290|
+----------+--------+-----+-------+-------+---------+---------+-------+



### Overall average comparison (Starbucks vs McDonald's, blended)

In [9]:
comparison_overall = combined.groupBy("Brand").agg(
    count("*").alias("Item_Count"),
    round(avg("Calories"), 1).alias("Avg_Calories"),
    round(avg("Fat_g"), 1).alias("Avg_Fat_g"),
    round(avg("Carbs_g"), 1).alias("Avg_Carbs_g"),
    round(avg("Fiber_g"), 1).alias("Avg_Fiber_g"),
    round(avg("Protein_g"), 1).alias("Avg_Protein_g"),
    round(avg("Sodium_mg"), 1).alias("Avg_Sodium_mg"),
    round(avg("Sugar_g"), 1).alias("Avg_Sugar_g"),
).orderBy("Brand")

comparison_overall.show(truncate=False)

+----------+----------+------------+---------+-----------+-----------+-------------+-------------+-----------+
|Brand     |Item_Count|Avg_Calories|Avg_Fat_g|Avg_Carbs_g|Avg_Fiber_g|Avg_Protein_g|Avg_Sodium_mg|Avg_Sugar_g|
+----------+----------+------------+---------+-----------+-----------+-------------+-------------+-----------+
|McDonald's|260       |368.3       |14.2     |47.3       |1.6        |13.3         |495.8        |29.4       |
|Starbucks |532       |222.9       |6.2      |85.4       |1.2        |7.5          |20.6         |33.0       |
+----------+----------+------------+---------+-----------+-----------+-------------+-------------+-----------+



### The key comparison: split by Brand AND item type

In [10]:
comparison_by_type = combined.groupBy("Brand", "item_type").agg(
    count("*").alias("Item_Count"),
    round(avg("Calories"), 1).alias("Avg_Calories"),
    round(avg("Fat_g"), 1).alias("Avg_Fat_g"),
    round(avg("Carbs_g"), 1).alias("Avg_Carbs_g"),
    round(avg("Fiber_g"), 1).alias("Avg_Fiber_g"),
    round(avg("Protein_g"), 1).alias("Avg_Protein_g"),
    round(avg("Sodium_mg"), 1).alias("Avg_Sodium_mg"),
    round(avg("Sugar_g"), 1).alias("Avg_Sugar_g"),
).orderBy("Brand", "item_type")

comparison_by_type.show(truncate=False)

+----------+---------+----------+------------+---------+-----------+-----------+-------------+-------------+-----------+
|Brand     |item_type|Item_Count|Avg_Calories|Avg_Fat_g|Avg_Carbs_g|Avg_Fiber_g|Avg_Protein_g|Avg_Sodium_mg|Avg_Sugar_g|
+----------+---------+----------+------------+---------+-----------+-----------+-------------+-------------+-----------+
|McDonald's|Food     |260       |368.3       |14.2     |47.3       |1.6        |13.3         |495.8        |29.4       |
|Starbucks |Drink    |419       |177.7       |2.7      |100.2      |0.7        |6.2          |20.6         |33.0       |
|Starbucks |Food     |113       |356.6       |16.4     |41.5       |2.8        |11.5         |NULL         |NULL       |
+----------+---------+----------+------------+---------+-----------+-----------+-------------+-------------+-----------+



### Find the most extreme individual items

In [13]:
print("--- 10 lowest-calorie items ---")
combined.select("Brand", "Item_Name", "Calories").where(col("Calories").isNotNull()).orderBy(col("Calories").asc()).show(10, truncate=False)

print("--- 10 highest-calorie items ---")
combined.select("Brand", "Item_Name", "Calories").where(col("Calories").isNotNull()).orderBy(col("Calories").desc()).show(10, truncate=False)

print("--- 10 highest-sodium items ---")
combined.select("Brand", "Item_Name", "Sodium_mg").where(col("Sodium_mg").isNotNull()).orderBy(col("Sodium_mg").desc()).show(10, truncate=False)

print("--- 10 highest-sugar items ---")
combined.select("Brand", "Item_Name", "Sugar_g").where(col("Sugar_g").isNotNull()).orderBy(col("Sugar_g").desc()).show(10, truncate=False)

--- 10 lowest-calorie items ---
+----------+-------------------+--------+
|Brand     |Item_Name          |Calories|
+----------+-------------------+--------+
|McDonald's|Dasani Water Bottle|0.0     |
|McDonald's|Iced Tea (Small)   |0.0     |
|McDonald's|Iced Tea (Medium)  |0.0     |
|McDonald's|Iced Tea (Large)   |0.0     |
|McDonald's|Iced Tea (Child)   |0.0     |
|McDonald's|Coffee (Small)     |0.0     |
|McDonald's|Coffee (Medium)    |0.0     |
|McDonald's|Coffee (Large)     |0.0     |
|McDonald's|Diet Coke (Small)  |0.0     |
|McDonald's|Diet Coke (Medium) |0.0     |
+----------+-------------------+--------+
only showing top 10 rows
--- 10 highest-calorie items ---
+----------+------------------------------------------------------------+--------+
|Brand     |Item_Name                                                   |Calories|
+----------+------------------------------------------------------------+--------+
|McDonald's|Chicken McNuggets (40 piece)                                |